# 🚀 خط أنابيب البيانات الهجين (Hybrid Data Pipeline) - المرحلة الأولى والثانية (المشروع النهائي الكامل)
### مقرر البيانات الضخمة (العملي) - جامعة الرازي | إشراف: م. عمر أبوسند
**التقنيات:** Python Batch Loading | Apache Spark | MongoDB | ELT | Data Quality | Idempotency | Indexes & Explain | Aggregations | Materialized Views | Scheduler | FastAPI

---
### 📋 فهرس خطوات التشغيل والمعالجة المتكاملة:
#### 🔹 المرحلة الأولى (المشروع النصفي - 18 درجة):
1. **الخطوة 0:** استخراج عينة اختبارية صغيرة قابلة لإعادة الإنتاج (`create_small_sample.py`).
2. **الخطوة 1:** استيراد الوحدات وتهيئة قاعدة بيانات MongoDB وتفعيل `Schema Validation` و `Unique Index`.
3. **الخطوة 2:** توليد معرف تشغيل فريد (`run_id`).
4. **الخطوة 3:** التوجيه التلقائي للمحرك (`File Router`) وفق حد 200MB.
5. **الخطوة 4:** التحميل الخام الأولي (`Raw Load`) إلى مجموعة `orders_raw` دون حذف أو تنظيف مسبق.
6. **الخطوة 5:** التحويل والتنظيف الآلي وتوثيق أثر التعديل (`Audit Trail`) والعزل والـ `Idempotent Upsert`.
7. **الخطوة 6:** استخراج المقاييس والتأكد من معادلة الاتساق وحفظ النتائج في `reports/results.json`.
8. **الخطوة 7:** إثبات الموثوقية (`Idempotency Proof`) بعدم تكرار السجلات عند إعادة التشغيل.

#### 🔹 المرحلة الثانية (المشروع النهائي - 7 درجات):
9. **الخطوة 8:** الاستعلامات المتقدمة وبناء الفهارس وتحليل خطط التنفيذ (`Indexes & Explain Analysis`).
10. **الخطوة 9:** خطوط أنابيب التجميع الإحصائي المتقدمة الخمسة (`Aggregation Pipelines`).
11. **الخطوة 10:** الجداول المادية والتحديث التزايدي الذكي (`Materialized Views & Incremental Refresh`).
12. **الخطوة 11:** الجدولة التلقائية وسجلات العمليات (`Background Scheduler & Job Logs`).
13. **الخطوة 12:** بوابة الخدمات السحابية (`FastAPI Gateway & Swagger UI`).
14. **الخطوة 13:** طاقم الاختبارات الآلية الشامل لكامل المشروع (`75/75 Unit & Integration Tests`).

## 0. إنشاء العينة الصغيرة للاختبار (القسم 6.1)
استخراج عينة صغيرة من الملف الضخم للاختبار والتأكد من قابلية إعادة الإنتاج دون فتح الملف يدوياً في Excel.

In [1]:
from src.create_small_sample import generate_sample

input_huge_file = 'data/orders_huge_mixed_quality.csv'
output_sample_file = 'data/orders_small_sample.csv'

# استخراج عينة من 100 ألف صف (أو 20 ألف صف) للاختبار السريع
# generate_sample(input_huge_file, output_sample_file, num_rows=2000000)
print('✅ خطوة استخراج العينة جاهزة ويمكن تفعيلها عند الحاجة.')

✅ خطوة استخراج العينة جاهزة ويمكن تفعيلها عند الحاجة.


## 1. استيراد المكتبات وتهيئة قاعدة البيانات (القسم 6.9)
نقوم باستيراد الوحدات البرمجية وتهيئة MongoDB وتطبيق **Schema Validation** و **Unique Index** على `order_id` لضمان عدم وجود تكرار.

In [2]:
import time
import uuid
import sys
import os
import json
from pymongo import MongoClient

from config.settings import MONGO_URI, MONGO_DB_NAME, COLLECTION_RAW, COLLECTION_VALIDATED, COLLECTION_QUARANTINE, SMALL_FILE_THRESHOLD_MB
from src.mongo_setup import setup_mongodb
from src.file_router import select_processing_engine
from src.batch_loader import load_batch
from src.spark_loader import load_spark
from src.elt_pipeline import run_elt
from src.metrics import generate_metrics

# تهيئة قاعدة البيانات والفهارس وقواعد التحقق من المخطط
print('⚙️ جاري تهيئة قاعدة بيانات MongoDB...')
setup_mongodb()

⚙️ جاري تهيئة قاعدة بيانات MongoDB...
⚙️ إعداد قاعدة بيانات MongoDB وتطبيق معايير القسم 6.9...
-> تم إنشاء المجموعة: orders_raw
-> تم إنشاء المجموعة: orders_validated
-> تم إنشاء المجموعة: orders_quarantine
-> إنشاء Unique Index على حقل order_id في مجموعة orders_validated...
-> ✅ تم تفعيل Schema Validation بنجاح على مجموعة orders_validated.
✅ اكتمل إعداد قاعدة البيانات بنجاح.



## 2. تحديد مسار ملف البيانات وتوليد Run ID فريد
نحدد مسار الملف المراد معالجته (مثلاً ملف الاختبار المعتمد `01_student_test_small.csv` أو العينة المستخرجة)، وننشئ معرّف تشغيل فريد `run_id`.

In [3]:
# اختر الملف الذي تريد تشغيله:
# 1. ملف الاختبار القياسي المعتمد (20,000 سجل):
# file_path = 'data/01_student_test_small.csv'

# 2. أو عينة الملف الكبير لاختبار محرك Spark:
file_path = 'data/orders_small_sample.csv'

run_id = f'run_{uuid.uuid4().hex[:8]}'
print(f'📍 ملف الإدخال: {file_path}')
print(f'🔑 معرف التشغيلة (Run ID): {run_id}')

📍 ملف الإدخال: data/orders_small_sample.csv
🔑 معرف التشغيلة (Run ID): run_30e27543


## 3. الموجه التلقائي للمحرك (File Router - القسم 6.2)
يقوم الموجه بفحص حجم الملف بناءً على الحد الفاصل المعتمد في الإعدادات `SMALL_FILE_THRESHOLD_MB = 200`:
- إذا كان الحجم $\le$ 200MB $\rightarrow$ يختار تلقائياً `python_batch`.
- إذا كان الحجم $>$ 200MB $\rightarrow$ يختار تلقائياً `pyspark`.

In [4]:
print('🔍 فحص حجم الملف لاختيار المحرك المناسب...')
engine = select_processing_engine(file_path)
assert engine in ['python_batch', 'pyspark'], 'خطأ: لم يتم تحديد المحرك بشكل سليم!'

🔍 فحص حجم الملف لاختيار المحرك المناسب...
حجم الملف: 839.03 MB
القرار: استخدام محرك (pyspark) لأن الحجم أكبر من 200MB


## 4. التحميل الخام وفق معمارية ELT (القسم 6.5)
يتم تحميل البيانات كاملة أولاً إلى مجموعة `orders_raw` دون تنظيف أو حذف مسبق، مع إضافة حقول التتبع:
- `run_id`، `source_file`، `source_row_number`، `ingested_at`، `engine_used`، و `raw_record`.

In [5]:
start_time_total = time.time()
print(f'📥 بدء عملية التحميل الخام باستخدام محرك [{engine}]...')

raw_loaded = 0
if engine == 'python_batch':
    raw_loaded = load_batch(run_id, file_path)
elif engine == 'pyspark':
    raw_loaded = load_spark(run_id, file_path)

print(f'\n✅ إجمالي السجلات التي وصلت إلى orders_raw: {raw_loaded} سجل.')
assert raw_loaded > 0, 'فشل: لم يتم تحميل أي بيانات خام!'

📥 بدء عملية التحميل الخام باستخدام محرك [pyspark]...
[run_30e27543] ⚡ بدء تحميل الملف الكبير عبر Apache Spark...
📊 عدد الـ Input Partitions المقروءة من الملف: 12
🔍 عرض خطة التنفيذ explain() لإثبات توزيع المهام:
== Physical Plan ==
*(1) Project [run_30e27543 AS run_id#35, data/orders_small_sample.csv AS source_file#36, null AS source_row_number#37, 2026-10-04 23:50:43.856402 AS ingested_at#38, pyspark AS engine_used#39, struct(order_id, order_id#0, order_date, order_date#1, status, status#2, customer_id, customer_id#3, customer_name, customer_name#4, customer_phone, customer_phone#5, customer_email, customer_email#6, city, city#7, district, district#8, delivery_type, delivery_type#9, delivery_cost, delivery_cost#10, payment_method, payment_method#11, payment_status, ... 9 more fields) AS raw_record#40]
+- FileScan csv [order_id#0,order_date#1,status#2,customer_id#3,customer_name#4,customer_phone#5,customer_email#6,city#7,district#8,delivery_type#9,delivery_cost#10,payment_method#11,paym

## 5. التحويل والتنظيف الآلي والعزل والـ Upsert (القسم 6.6 و 6.7 و 6.8 و 6.10)
تطبيق قواعد الجودة والتنظيف الآلي:
- **السجلات السليمة:** تنتقل إلى `orders_validated` مباشرة.
- **السجلات القابلة للتصحيح:** تُصحح وتُحفظ مع أثر التصحيح الكامل (`Audit Trail`) في `orders_validated` عبر `Upsert`.
- **السجلات ذات الأخطاء الجوهرية:** تُعزل في `orders_quarantine` مع تسجيل كافة أسباب العزل والسجل الخام.

In [7]:
print('🛠️ بدء عمليات التنظيف، العزل، وتطبيق الـ Idempotent Upsert...')
elt_counters, elt_elapsed = run_elt(run_id)

print(f'\n📊 نتائج معالجة ELT للتشغيلة [{run_id}]:')
print(f'  - المقروء: {elt_counters["read"]}')
print(f'  - السليم: {elt_counters["valid"]}')
print(f'  - المصحح (مع Audit Trail): {elt_counters["corrected"]}')
print(f'  - المعزول (Quarantine): {elt_counters["quarantined"]}')
print(f'  - المدخل الجديد (Inserted): {elt_counters["inserted"]}')
print(f'  - المحدث (Updated): {elt_counters["updated"]}')
print(f'  - غير المتغير (Unchanged): {elt_counters["unchanged"]}')

🛠️ بدء عمليات التنظيف، العزل، وتطبيق الـ Idempotent Upsert...
[run_30e27543] 2. بدء عملية التنظيف (ELT) والـ Upsert بنمط تدفقي متوازي مع حماية نقاط الحفظ (Checkpointing)...

🔄 [استئناف تلقائي - Checkpoint Found]
-> تم العثور على آخر مكان وصلت له المعالجة للتشغيلة [run_30e27543]:
-> السجلات المنجزة سابقاً: 1,030,000 سجل (سليم: 0 | مصحح: 946,430 | معزول: 83,570)
-> آخر معرّف معالج (Checkpoint ID): 6ac2bc4ae5f8771b4a6d970c
-> تم تفادي إعادة معالجة ما سبق، وسيتم إكمال باقي السجلات فوراً! 🚀

-> تم إنجاز: 1,040,000 سجل | السرعة: 4,220 سجل/ثانية | سليم: 0 | مصحح: 955,640 | معزول: 84,360
-> تم إنجاز: 1,050,000 سجل | السرعة: 4,219 سجل/ثانية | سليم: 0 | مصحح: 964,841 | معزول: 85,159
-> تم إنجاز: 1,060,000 سجل | السرعة: 4,215 سجل/ثانية | سليم: 0 | مصحح: 974,060 | معزول: 85,940
-> تم إنجاز: 1,070,000 سجل | السرعة: 4,208 سجل/ثانية | سليم: 0 | مصحح: 983,229 | معزول: 86,771
-> تم إنجاز: 1,080,000 سجل | السرعة: 4,211 سجل/ثانية | سليم: 0 | مصحح: 992,435 | معزول: 87,565
-> تم إنجاز: 1,090,000 سجل | السر

## 6. استخراج المقاييس والتأكد من معادلة الاتساق (القسم 6.11 و 6.12)
التحقق من شرط القبول ومعادلة الاتساق الأساسية:
$$\text{run\_raw\_count} = \text{run\_valid\_count} + \text{run\_corrected\_count} + \text{run\_quarantine\_count}$$
وحفظ التقرير الشامل في `reports/results.json`.

In [8]:
print('📊 إنشاء وتصدير تقرير المقاييس (Metrics)...')
total_seconds = time.time() - start_time_total
metrics = generate_metrics(run_id, file_path, engine, raw_loaded, elt_counters, total_seconds)

print('\n✅ التقرير النهائي للتشغيلة الحالية:')
print(json.dumps(metrics, indent=4, ensure_ascii=False))

# تأكيد اختبار الاتساق
assert metrics['idempotency_check'] == 'PASS', 'تحذير: فشل معادلة الاتساق!'

📊 إنشاء وتصدير تقرير المقاييس (Metrics)...
[run_30e27543] 3. إنشاء تقرير المقاييس (Metrics)...
✅ تم حفظ التقارير في reports\results.json

✅ التقرير النهائي للتشغيلة الحالية:
{
    "run_id": "run_30e27543",
    "file_name": "orders_small_sample.csv",
    "file_size_mb": 839.03,
    "engine_used": "pyspark",
    "rows_read": 2000000,
    "raw_loaded": 2000000,
    "valid_count": 0,
    "corrected_count": 1837922,
    "quarantine_count": 162078,
    "elapsed_seconds": 874.13,
    "throughput": 2287.99,
    "batch_size_or_partitions": 100,
    "error_case_counts": {
        "negative_quantity": 13802,
        "missing_order_id": 13961,
        "MISSING_ORDER_ID": 13961,
        "missing_customer_id": 14139,
        "MISSING_CUSTOMER_ID": 14139,
        "invalid_date_impossible": 36320,
        "INVALID_IMPOSSIBLE_DATE": 36320,
        "invalid_phone_too_short": 14323,
        "missing_item_sku": 13932,
        "email_missing_domain": 14035,
        "multiple_conflicting_errors": 13953,
   

## 7. اختبار الموثوقية وعدم التكرار عملياً (Idempotency & Upsert Proof - القسم 6.10)
نقوم بإعادة تشغيل نفس عملية الـ ELT على نفس البيانات لإثبات أن عدد السجلات التجارية في `orders_validated` لا يزيد ولا يُنشئ أي سجلات مكررة.

In [11]:
print('🔄 اختبار Idempotency: إعادة تشغيل نفس بيانات run_id الحالية...')
client = MongoClient(MONGO_URI)
db = client[MONGO_DB_NAME]
count_before = db[COLLECTION_VALIDATED].count_documents({})

# إعادة تشغيل ELT
re_counters, _ = run_elt(run_id)
count_after = db[COLLECTION_VALIDATED].count_documents({})

print(f'\nعدد السجلات في orders_validated قبل الإعادة: {count_before}')
print(f'عدد السجلات في or ders_validated بعد الإعادة: {count_after}')
print(f'السجلات المضافة في الإعادة (Inserted): {re_counters["inserted"]}')
print(f'السجلات غير المتغيرة (Unchanged): {re_counters["unchanged"]}')

assert count_before == count_after, '❌ فشل اختبار Idempotency: زادت عدد السجلات في قاعدة البيانات!'
assert re_counters['inserted'] == 0, '❌ تم إدخال سجلات جديدة مكررة بالخطأ!'
print('🎉 تم إثبات الـ Idempotency بنجاح تام! لم تنشأ أي سجلات مكررة (0 Duplicates).')
client.close()

🔄 اختبار Idempotency: إعادة تشغيل نفس بيانات run_id الحالية...
[run_30e27543] 2. بدء عملية التنظيف (ELT) والـ Upsert بنمط تدفقي متوازي مع حماية نقاط الحفظ (Checkpointing)...

🔄 [استئناف تلقائي - Checkpoint Found]
-> تم العثور على آخر مكان وصلت له المعالجة للتشغيلة [run_30e27543]:
-> السجلات المنجزة سابقاً: 1,715,000 سجل (سليم: 0 | مصحح: 1,576,045 | معزول: 138,955)
-> آخر معرّف معالج (Checkpoint ID): 6ac2bc5e6143f201322daa08
-> تم تفادي إعادة معالجة ما سبق، وسيتم إكمال باقي السجلات فوراً! 🚀

-> تم إنجاز: 1,725,000 سجل | السرعة: 5,964 سجل/ثانية | سليم: 0 | مصحح: 1,585,225 | معزول: 139,775
-> تم إنجاز: 1,740,000 سجل | السرعة: 5,965 سجل/ثانية | سليم: 0 | مصحح: 1,599,041 | معزول: 140,959
-> تم إنجاز: 1,755,000 سجل | السرعة: 5,963 سجل/ثانية | سليم: 0 | مصحح: 1,612,830 | معزول: 142,170
-> تم إنجاز: 1,775,000 سجل | السرعة: 5,978 سجل/ثانية | سليم: 0 | مصحح: 1,631,222 | معزول: 143,778
-> تم إنجاز: 1,790,000 سجل | السرعة: 5,985 سجل/ثانية | سليم: 0 | مصحح: 1,645,005 | معزول: 144,995
-> تم إنجاز: 1

## 8. الاستعلامات المتقدمة ودراسة الأداء قبل وبعد الفهارس (Indexes & Explain Analysis)
بناء 4 فهارس متخصصة تشمل فهارس مركبة (Compound Indexes) وتشغيل 5 استعلامات عملية مع مقارنة مقاييس `executionStats`:
- نوع الفحص: `COLLSCAN` (مسح الجدول كاملاً) مقابل `IXSCAN` (فحص الفهرس فائق السرعة).
- عدد الوثائق المفحوصة `totalDocsExamined`.
- زمن التنفيذ بالمللي ثانية `executionTimeMillis`.

In [12]:
from src.indexes_queries import create_indexes, run_explain_analysis, query_customer_orders, query_orders_by_date_and_status

# 1. إنشاء الفهارس المطلوبة
indexes_created = create_indexes()
print('📌 الفهارس المفعلة في المجموعة:', indexes_created)

# 2. إجراء تحليل مقارنة الأداء (Before vs After Indexes) على 3 استعلامات
print('\n🔬 نتائج تحليل خطط التنفيذ (Explain Plan Comparison):')
analysis = run_explain_analysis()
for q_name, stats in analysis.items():
    print(f'\n🔹 الاستعلام [{q_name}]:')
    print(f'   - بدون فهرس: المرحلة={stats["without_index"]["stage"]} | وثائق مفحوصة={stats["without_index"]["total_docs_examined"]} | الزمن={stats["without_index"]["execution_time_ms"]} ms')
    print(f'   - مع الفهرس:  المرحلة={stats["with_index"]["stage"]} | مفاتيح مفحوصة={stats["with_index"]["total_keys_examined"]} | وثائق مفحوصة={stats["with_index"]["total_docs_examined"]} | الزمن={stats["with_index"]["execution_time_ms"]} ms')
    print(f'   - مؤشر التحسين: {stats["improvement"]}')

📌 الفهارس المفعلة في المجموعة: [{'name': 'idx_customer_date', 'keys': [('customer_id', 1), ('order_date', -1)], 'type': 'Compound Index', 'purpose': 'تسريع استعلام سجل طلبات العميل وتاريخها مرتبة زمنياً من الأحدث للأقدم.'}, {'name': 'idx_date_status', 'keys': [('order_date', 1), ('status', 1)], 'type': 'Compound Index', 'purpose': 'تسريع استعلامات التقارير الزمنية المفلترة بحالة الطلب (مثل الطلبات المكتملة في فترة محددة).'}, {'name': 'idx_total_amount', 'keys': [('total_amount', -1)], 'type': 'Single Index', 'purpose': 'تسريع استعلامات الطلبات ذات القيمة المالية المرتفعة وترتيب المبيعات تنازلياً.'}, {'name': 'idx_city_delivery', 'keys': [('city', 1), ('delivery_type', 1)], 'type': 'Compound Index', 'purpose': 'تسريع استعلامات التحليل الجغرافي ونوع التوصيل حسب كل مدينة.'}]

🔬 نتائج تحليل خطط التنفيذ (Explain Plan Comparison):

🔹 الاستعلام [استعلام سجل طلبات العميل (Customer Orders Query)]:
   - بدون فهرس: المرحلة=SORT | وثائق مفحوصة=1825067 | الزمن=1240 ms
   - مع الفهرس:  المرحلة=FETCH

## 9. تقارير التجميع الإحصائي المتقدمة (5 Aggregation Pipelines)
تنفيذ العمليات الإحصائية التحليلية الخمس المطلوبة عبر محرك التجميع الخاص بـ MongoDB:
1. إجمالي المبيعات ومتوسط قيمة الطلب حسب المدينة (`sales_by_city`).
2. أفضل 5 منتجات مبيعاً من حيث الإيرادات والكميات (`top_products`).
3. كبار العملاء وأعلى إنفاقاً (`top_customers`).
4. تحليل المبيعات حسب الفترة الزمنية السنوية/الشهرية (`sales_by_period`).
5. توزيع الطلبات والإيرادات حسب حالة الطلب (`orders_by_status`).

In [13]:
from src.aggregations import (
    aggregate_sales_by_city,
    aggregate_top_products,
    aggregate_top_customers,
    aggregate_sales_by_period,
    aggregate_orders_by_status
)

print('🏙️ 1. المبيعات حسب المدينة (أعلى 5):')
for item in aggregate_sales_by_city(limit=5):
    print(f'   {item["city"]}: إجمالي المبيعات = {item["total_sales"]:,.2f} | عدد الطلبات = {item["order_count"]}')

print('\n📦 2. أفضل المنتجات مبيعاً:')
for item in aggregate_top_products(limit=5):
    print(f'   {item["product_name"]}: إيرادات = {item["total_revenue"]:,.2f} | إجمالي الكميات = {item["total_quantity"]}')

print('\n👥 3. كبار العملاء:')
for item in aggregate_top_customers(limit=5):
    print(f'   {item["customer_name"]} ({item["customer_id"][-8:]}): إجمالي الإنفاق = {item["total_spent"]:,.2f} | الطلبات = {item["order_count"]}')

print('\n📅 4. المبيعات حسب الفترة (أحدث الفترات):')
for item in aggregate_sales_by_period(limit=5):
    print(f'   {item["period"]}: مبيعات = {item["total_sales"]:,.2f} | طلبات = {item["order_count"]}')

print('\n📊 5. توزيع الطلبات حسب الحالة:')
for item in aggregate_orders_by_status():
    print(f'   {item["order_status"]}: عدد الطلبات = {item["order_count"]} | إجمالي القيمة = {item["total_value"]:,.2f}')

🏙️ 1. المبيعات حسب المدينة (أعلى 5):
   صنعاء: إجمالي المبيعات = 47,380,742,500.00 | عدد الطلبات = 183818
   لحج: إجمالي المبيعات = 47,258,390,000.00 | عدد الطلبات = 183197
   عدن: إجمالي المبيعات = 47,050,153,500.00 | عدد الطلبات = 182855
   الحديدة: إجمالي المبيعات = 46,937,443,000.00 | عدد الطلبات = 182027
   المكلا: إجمالي المبيعات = 46,884,480,000.00 | عدد الطلبات = 182706

📦 2. أفضل المنتجات مبيعاً:
   هاتف سامسونج A54: إيرادات = 1,453,082,000.00 | إجمالي الكميات = 6607
   قرص SSD محمول: إيرادات = 705,807,500.00 | إجمالي الكميات = 6505
   سماعات رأس: إيرادات = 187,981,000.00 | إجمالي الكميات = 6657
   محول HDMI: إيرادات = 66,379,000.00 | إجمالي الكميات = 6609
   فأرة لاسلكية: إيرادات = 59,219,500.00 | إجمالي الكميات = 6601

👥 3. كبار العملاء:
   يوسف طارق (ل-237376): إجمالي الإنفاق = 2,247,500.00 | الطلبات = 1
   أروى محمد (ل-446510): إجمالي الإنفاق = 2,234,000.00 | الطلبات = 1
   أحمد محمد (-1168454): إجمالي الإنفاق = 2,225,000.00 | الطلبات = 1
   فاطمة أحمد (ل-981798): إجمالي ا

## 10. الجداول المادية والتحديث التزايدي فائق السرعة (Materialized Views)
بناء جدولين ماديين لتقديم استجابة لحظية للتقارير واللوحات التحليلية مع آلية **التحديث التزايدي الذكي (Incremental Refresh)** عبر `$inc` و `$set` دون الحاجة لإعادة مسح أو إعادة بناء الجدول بالكامل:
1. جدول ملخص المبيعات اليومية (`daily_sales_summary`).
2. جدول ملخص أفضل المنتجات مبيعاً (`top_products_summary`).

In [14]:
from src.materialized_views import refresh_all_materialized_views, get_daily_sales_summary, get_top_products_summary

# تنفيذ التحديث (تحديث تزايدي إن وجدت بيانات سابقة، أو تحديث شامل أولي)
print('⚡ جاري تحديث الجداول المادية...')
mv_result = refresh_all_materialized_views()
print(f'✅ اكتمل التحديث: {mv_result}')

# استعراض عينة من البيانات المحدثة داخل الجداول المادية
daily_sample = get_daily_sales_summary(limit=3)
print('\n📅 عينة من جدول ملخص المبيعات اليومية (daily_sales_summary):')
for row in daily_sample:
    print(f'   التاريخ: {row["date"]} | المبيعات: {row["total_sales"]:,.2f} | الطلبات: {row["order_count"]}')

top_sample = get_top_products_summary(limit=3)
print('\n🏆 عينة من جدول ملخص أفضل المنتجات (top_products_summary):')
for row in top_sample:
    print(f'   المنتج: {row["product_name"]} | الإيراد: {row["total_revenue"]:,.2f} | الكميات: {row["total_quantity"]}')

⚡ جاري تحديث الجداول المادية...
✅ اكتمل التحديث: {'timestamp': '2026-10-05T00:19:45.916064', 'incremental': True, 'daily_sales_summary': {'view_name': 'daily_sales_summary', 'status': 'SUCCESS', 'refresh_mode': 'full', 'days_updated': 121, 'last_synced_id': '6ac2bf031c3fb5d1d3053479', 'elapsed_seconds': 7.781}, 'top_products_summary': {'view_name': 'top_products_summary', 'status': 'SUCCESS', 'refresh_mode': 'full', 'products_updated': 6, 'records_processed': 50000, 'last_synced_id': '6ac2bc77476b99a19425cc23', 'elapsed_seconds': 0.677}}

📅 عينة من جدول ملخص المبيعات اليومية (daily_sales_summary):
   التاريخ: 2025-02-28 | المبيعات: 4,000,673,000.00 | الطلبات: 15366
   التاريخ: 2025-04-17 | المبيعات: 3,995,437,000.00 | الطلبات: 15329
   التاريخ: 2025-04-18 | المبيعات: 3,985,536,000.00 | الطلبات: 15270

🏆 عينة من جدول ملخص أفضل المنتجات (top_products_summary):
   المنتج: هاتف سامسونج A54 | الإيراد: 7,299,246,000.00 | الكميات: 33132
   المنتج: قرص SSD محمول | الإيراد: 3,622,361,000.00 | ا

## 11. إدارة المهام المجدولة وسجلات العمليات (Scheduler & Job Audit Logs)
تنظيم المهام الخلفية المجدولة مع إمكانية التشغيل اليدوي المباشر وتوثيق كامل تفاصيل التشغيل في مجموعة `job_logs` في MongoDB:
- `refresh_materialized_views_job`: تحديث الجداول المادية دورياً.
- `periodic_kpi_report_job`: حساب وتخزين مؤشرات الأداء وحجم البيانات دورياً.

In [15]:
from src.scheduler import run_job, get_job_logs

print('⏱️ تجربة تشغيل المهام المجدولة يدوياً (Manual Trigger):')
res1 = run_job('refresh_materialized_views_job')
print(f'   - مهمة تحديث الجداول المادية: {res1["status"]} (استغرقت {res1["duration_ms"]} ms)')

res2 = run_job('periodic_kpi_report_job')
print(f'   - مهمة تقرير مؤشرات الأداء KPI: {res2["status"]} (استغرقت {res2["duration_ms"]} ms)')

print('\n📋 أحدث سجلات التنفيذ المحفوظة في MongoDB (job_logs):')
logs = get_job_logs(limit=4)
for log in logs:
    print(f'   [{log.get("timestamp")}] المهمة: {log.get("job_name")} | الحالة: {log.get("status")} | التفاصيل: {log.get("message")}')

⏱️ تجربة تشغيل المهام المجدولة يدوياً (Manual Trigger):
   - مهمة تحديث الجداول المادية: SUCCESS (استغرقت 734.12 ms)
   - مهمة تقرير مؤشرات الأداء KPI: SUCCESS (استغرقت 10155.49 ms)

📋 أحدث سجلات التنفيذ المحفوظة في MongoDB (job_logs):
   [None] المهمة: periodic_kpi_report_job | الحالة: SUCCESS | التفاصيل: None
   [None] المهمة: refresh_materialized_views_job | الحالة: SUCCESS | التفاصيل: None


## 12. واجهة برمجة التطبيقات والتوثيق التفاعلي (FastAPI Gateway & Swagger UI)
توفير واجهة RESTful API كاملة تغطي 10 نقاط نهاية (Endpoints) متوافقة 100% مع مواصفات المشروع النهائي:
- فحص الصحة والمؤشرات: `/api/health`, `/api/metrics`
- تقارير التجميع: `/api/reports/sales-by-city`, `/api/reports/top-products`, `/api/reports/top-customers`, `/api/reports/sales-by-period`, `/api/reports/orders-by-status`
- الجداول المادية: `/api/views/daily-sales`, `/api/views/top-products`, `/api/views/refresh` (POST)
- التوثيق التفاعلي الكامل متاح على: `http://127.0.0.1:8000/docs` (Swagger UI).

In [16]:
from fastapi.testclient import TestClient
from src.api import app

# اختبار واجهة الـ API مباشرة داخل بيئة بايثون للتأكد من استجابة النقاط
client = TestClient(app)

health_resp = client.get('/health')
print('🏥 فحص حالة الخدمة (/health):', health_resp.json())

metrics_resp = client.get('/metrics')
print('\n📈 مؤشرات الأداء الحالية (/metrics):')
for k, v in list(metrics_resp.json().items())[:6]:
    print(f'   - {k}: {v}')

sales_city_resp = client.get('/aggregations/sales_by_city?limit=3')
print('\n🏙️ استجابة تقرير المبيعات حسب المدينة (/aggregations/sales_by_city):')
print(f'   كود الاستجابة: {sales_city_resp.status_code} | عدد السجلات: {sales_city_resp.json().get("records_count", 0)}')

print('\n🌐 لتشغيل خادم الـ API الحي، قم بتنفيذ الأمر التالي في الطرفية:')
print('   python -m uvicorn src.api:app --host 127.0.0.1 --port 8000 --reload')
print('   ثم افتح المتصفح على: http://127.0.0.1:8000/docs')

🏥 فحص حالة الخدمة (/health): {'status': 'UP', 'service': 'Big Data Pipeline Unified API (FastAPI)', 'database': 'CONNECTED', 'database_latency_ms': 29.15, 'timestamp': '2026-10-05T00:19:57.908887', 'scheduler_status': 'RUNNING'}

📈 مؤشرات الأداء الحالية (/metrics):
   - orders_raw_count: 2000000
   - orders_validated_count: 1804296
   - orders_quarantine_count: 280907
   - consistency_equation_status: CHECK_REQUIRED
   - latest_pipeline_run: {}

🏙️ استجابة تقرير المبيعات حسب المدينة (/aggregations/sales_by_city):
   كود الاستجابة: 200 | عدد السجلات: 3

🌐 لتشغيل خادم الـ API الحي، قم بتنفيذ الأمر التالي في الطرفية:
   python -m uvicorn src.api:app --host 127.0.0.1 --port 8000 --reload
   ثم افتح المتصفح على: http://127.0.0.1:8000/docs


## 13. طاقم الاختبارات الآلية الشامل لكامل المشروع (75/75 Tests - 100% Success)
تشغيل كافة اختبارات الـ Unit Tests واختبارات التكامل لكامل المرحلة الأولى والثانية (75 اختباراً):
- اختبارات التصنيف والتنظيف ومعادلة الاتساق والـ Upsert (المرحلة الأولى - 62 اختباراً).
- اختبارات الفهارس، التجميعات، الجداول المادية، المجدول، والـ API (المرحلة الثانية - 13 اختباراً).

In [17]:
import subprocess
import sys

print('🧪 جاري تشغيل طاقم الاختبارات الشامل لكامل المشروع (75 اختباراً)...')
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True, encoding='utf-8', errors='ignore'
)
print(test_run.stdout)
if test_run.returncode == 0:
    print('🏆 تهانينا! جميع الاختبارات الـ 75 اجتازت الفحص بنجاح تام 100% (25/25 علامة كاملة)!')
else:
    print('⚠️ تفاصيل الأخطاء إن وجدت:')
    print(test_run.stderr)

🧪 جاري تشغيل طاقم الاختبارات الشامل لكامل المشروع (75 اختباراً)...
============================= test session starts =============================
platform win32 -- Python 3.12.8, pytest-9.0.2, pluggy-1.6.0 -- c:\Users\USER\AppData\Local\Programs\Python\Python312\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\USER\OneDrive - balal\Desktop\midterm-data-pipline
configfile: pytest.ini
plugins: anyio-4.11.0, dash-3.3.0, Faker-38.0.0, timeout-2.4.0
collecting ... collected 75 items

tests/test_classification.py::TestClassificationOutcomes::test_clean_record_classification PASSED [  1%]
tests/test_classification.py::TestClassificationOutcomes::test_correctable_record_classification PASSED [  2%]
tests/test_classification.py::TestClassificationOutcomes::test_quarantined_record_classification PASSED [  4%]
tests/test_classification.py::TestQuarantineReasonsCoverage::test_missing_order_id PASSED [  5%]
tests/test_classification.py::TestQuarantineReasonsCoverage::test_missing_customer_id P